# Long-Term Agent Memory in Neo4j (a graph instead of Mem0)

Same **ticket agent** as `../mem0/long_memory_mem0.ipynb`, but **Neo4j replaces both Postgres and Mem0**. One database holds two separate graphs:

| Graph | Purpose | Labels |
|---|---|---|
| **Ticket graph** | exact, current ticket *state* (looked up by tools) | `User`, `Ticket`, `Event`, `Person`, `Counter` |
| **Memory graph** | long-term *context*: what we know about the user and past chats | `Entity`, `Episode` |

What Mem0's `add()` / `search()` did for us, we now build ourselves on top of Neo4j:

| Mem0 did this | Here we do it with |
|---|---|
| fact extraction | an LLM with structured output -> `(subject, predicate, object)` triples |
| dedupe / update / delete | same LLM call sees the known facts and returns the outdated ones; we close them (`active=false`, `valid_to`) instead of deleting |
| entity merging ("Ravi" = "Ravi K") | embedding similarity on a Neo4j **vector index** |
| `search()` | vector search for relevant entities, then a **graph traversal** for their facts |

## The memory graph model

```
(:Entity {id, user_id, name, type, embedding})
(:Entity)-[:REL {id, predicate, fact, active, valid_from, valid_to}]->(:Entity)
(:Episode {id, user_id, text, created_at})-[:MENTIONS]->(:Entity)
```

Example: `(Shiva)-[:REL {predicate:'WORKS_ON'}]->(Payments team)`, `(Refund API docs)-[:REL {predicate:'WAITING_ON'}]->(Ravi)`.
All facts use one relationship type `REL` with a `predicate` property (dynamic relationship types would need the APOC plugin).
Outdated facts are kept with `active=false`, so you also get a history of how the user's world changed.

**Needs:** Docker running and `OPENAI_API_KEY` in your `.env` (no Mem0 key any more). Neo4j settings (`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`) default to the container in `docker-compose.yml`.

In [ ]:
%pip install -qU neo4j langchain langchain-openai pydantic python-dotenv

## 0. API keys

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

## 1. Neo4j in Docker (ticket state *and* long-term memory)

`docker-compose.yml` (next to this notebook) starts Neo4j 5 as container `ticket-neo4j`:
Bolt on host port **7688**, Neo4j Browser on **http://localhost:7475** (login `neo4j` / `ticket_pass123`).
Open the Browser and run `MATCH (n) RETURN n` after the chat to *see* the graph.

> Rule of thumb: **Ticket graph = what is true right now. Memory graph = what we know about the user and what we talked about.**

In [ ]:
import subprocess, time
from neo4j import GraphDatabase

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7688")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "ticket_pass123")

# Start the container only when using the default local instance (no-op if already running)
if NEO4J_URI.startswith("bolt://localhost"):
    subprocess.run(["docker", "compose", "up", "-d"], check=True)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def cypher(query: str, **params) -> list[dict]:
    """Run a Cypher query and return the rows as plain dicts."""
    records, _, _ = driver.execute_query(query, params)
    return [r.data() for r in records]


# Wait until Neo4j accepts connections
for _ in range(40):
    try:
        cypher("RETURN 1")
        print("Neo4j is ready")
        break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError("Neo4j did not start - check `docker compose logs`")

### Schema (constraints and the vector index)

Neo4j is schema-optional, so we add constraints ourselves: unique ticket ids, one node per user/person, and one `Entity` per `(user, name)` (stored as a single `id` like `shiva:ravi`). A `Counter` node gives auto-increment ticket ids.

The **vector index** on `Entity.embedding` (1536 dims = `text-embedding-3-small`) is what lets us find the entities relevant to a message and merge near-duplicate names.

In [ ]:
EMBED_DIMS = 1536

for stmt in [
    "CREATE CONSTRAINT ticket_id IF NOT EXISTS FOR (t:Ticket) REQUIRE t.id IS UNIQUE",
    "CREATE CONSTRAINT user_id IF NOT EXISTS FOR (u:User) REQUIRE u.id IS UNIQUE",
    "CREATE CONSTRAINT person_name IF NOT EXISTS FOR (p:Person) REQUIRE p.name IS UNIQUE",
    "CREATE CONSTRAINT counter_name IF NOT EXISTS FOR (c:Counter) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE",
    "CREATE CONSTRAINT episode_id IF NOT EXISTS FOR (e:Episode) REQUIRE e.id IS UNIQUE",
    f"""CREATE VECTOR INDEX entity_embedding IF NOT EXISTS FOR (e:Entity) ON e.embedding
        OPTIONS {{indexConfig: {{`vector.dimensions`: {EMBED_DIMS}, `vector.similarity_function`: 'cosine'}}}}""",
]:
    cypher(stmt)
cypher("CALL db.awaitIndexes(60)")
print("Constraints and vector index ready")

## 2. Ticket tools

Same five tools as the Postgres version, plus a sixth (`block_ticket`) that uses the graph: it links a ticket to the person it is waiting on. Every tool talks to Neo4j - the LLM never "remembers" ticket status, it **looks it up**.

In [ ]:
import json
from typing import Literal
from langchain_core.tools import tool

Status = Literal["open", "in_progress", "blocked", "done"]
Priority = Literal["low", "medium", "high"]

USER_ID = "shiva"


def _log(ticket_id: int, event: str):
    cypher(
        "MATCH (t:Ticket {id: $id}) CREATE (t)-[:HAS_EVENT]->(:Event {text: $event, created_at: datetime()})",
        id=ticket_id, event=event,
    )


@tool
def create_ticket(title: str, description: str = "", priority: Priority = "medium") -> str:
    """Create a new ticket for a task assigned to the user. Returns the new ticket id."""
    row = cypher(
        """
        MERGE (c:Counter {name: 'ticket'})
        ON CREATE SET c.value = 0
        SET c.value = c.value + 1
        WITH c
        MERGE (u:User {id: $user})
        CREATE (u)-[:ASSIGNED]->(t:Ticket {
            id: c.value, title: $title, description: $description,
            status: 'open', priority: $priority,
            created_at: datetime(), updated_at: datetime()})
        RETURN t.id AS id
        """,
        user=USER_ID, title=title, description=description, priority=priority,
    )[0]
    _log(row["id"], f"created (priority={priority})")
    return f"Created ticket #{row['id']}: {title}"


@tool
def list_tickets(status: str = "") -> str:
    """List tickets. Pass a status (open, in_progress, blocked, done) to filter; empty returns all."""
    rows = cypher(
        """
        MATCH (t:Ticket)
        WHERE $status = '' OR t.status = $status
        OPTIONAL MATCH (t)-[:BLOCKED_BY]->(p:Person)
        RETURN t.id AS id, t.title AS title, t.status AS status, t.priority AS priority, p.name AS blocked_by
        ORDER BY id
        """,
        status=status,
    )
    return json.dumps(rows, default=str) if rows else "No tickets found."


@tool
def update_ticket_status(ticket_id: int, status: Status) -> str:
    """Change the status of a ticket. Leaving 'blocked' clears who it was blocked by."""
    rows = cypher(
        """
        MATCH (t:Ticket {id: $id})
        SET t.status = $status, t.updated_at = datetime()
        WITH t
        OPTIONAL MATCH (t)-[b:BLOCKED_BY]->()
        WHERE $status <> 'blocked'
        DELETE b
        RETURN DISTINCT t.title AS title
        """,
        id=ticket_id, status=status,
    )
    if not rows:
        return f"Ticket #{ticket_id} does not exist."
    _log(ticket_id, f"status -> {status}")
    return f"Ticket #{ticket_id} ('{rows[0]['title']}') is now {status}."


@tool
def block_ticket(ticket_id: int, blocked_by: str) -> str:
    """Mark a ticket as blocked and record the person (or team) it is waiting on."""
    rows = cypher(
        """
        MATCH (t:Ticket {id: $id})
        SET t.status = 'blocked', t.updated_at = datetime()
        MERGE (p:Person {name: $who})
        MERGE (t)-[:BLOCKED_BY]->(p)
        RETURN t.title AS title
        """,
        id=ticket_id, who=blocked_by,
    )
    if not rows:
        return f"Ticket #{ticket_id} does not exist."
    _log(ticket_id, f"status -> blocked (waiting on {blocked_by})")
    return f"Ticket #{ticket_id} ('{rows[0]['title']}') is blocked, waiting on {blocked_by}."


@tool
def add_ticket_note(ticket_id: int, note: str) -> str:
    """Add a progress note to a ticket's log."""
    if not cypher("MATCH (t:Ticket {id: $id}) RETURN t.id AS id", id=ticket_id):
        return f"Ticket #{ticket_id} does not exist."
    _log(ticket_id, f"note: {note}")
    return f"Note added to ticket #{ticket_id}."


@tool
def get_ticket_history(ticket_id: int) -> str:
    """Show a ticket's details and its full event log."""
    rows = cypher(
        """
        MATCH (t:Ticket {id: $id})
        OPTIONAL MATCH (t)-[:BLOCKED_BY]->(p:Person)
        OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:Event)
        WITH t, p, e ORDER BY e.created_at
        RETURN properties(t) AS ticket, p.name AS blocked_by,
               [x IN collect(e) WHERE x IS NOT NULL | {event: x.text, created_at: x.created_at}] AS log
        """,
        id=ticket_id,
    )
    if not rows:
        return f"Ticket #{ticket_id} does not exist."
    return json.dumps(rows[0], default=str)


TOOLS = [create_ticket, list_tickets, update_ticket_status, block_ticket, add_ticket_note, get_ticket_history]

In [ ]:
TICKETS_QUERY = """
MATCH (t:Ticket)
WHERE $include_done OR t.status <> 'done'
OPTIONAL MATCH (t)-[:BLOCKED_BY]->(p:Person)
RETURN t.id AS id, t.title AS title, t.status AS status, t.priority AS priority, p.name AS blocked_by
ORDER BY id
"""


def tickets_snapshot() -> str:
    """Compact view of all not-done tickets - injected into every prompt so the agent always sees live state."""
    rows = cypher(TICKETS_QUERY, include_done=False)
    if not rows:
        return "(no active tickets)"
    return "\n".join(
        f"#{r['id']} [{r['status']}, {r['priority']}] {r['title']}" + (f" (blocked by {r['blocked_by']})" if r["blocked_by"] else "")
        for r in rows
    )


def show_tickets():
    rows = cypher(TICKETS_QUERY, include_done=True)
    for r in rows:
        extra = f"  <- blocked by {r['blocked_by']}" if r["blocked_by"] else ""
        print(f"#{r['id']:<3} {r['status']:<12} {r['priority']:<7} {r['title']}{extra}")
    if not rows:
        print("(no tickets)")


def show_tool_calls(new_messages):
    """Print the tools the agent called this turn, and return them as a short text."""
    calls = []
    for m in new_messages:
        for tc in getattr(m, "tool_calls", None) or []:
            args = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
            calls.append(f"{tc['name']}({args})")
    for c in calls:
        print(f"   [tool] {c}")
    return calls

## 3. Models

An LLM for the agent and for fact extraction, and an embedding model for the vector index. Nothing else - no Mem0 client.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-5-nano")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

USER_NAME = "Shiva"   # how the speaker is named in the memory graph

## 4. The memory layer on Neo4j

Four pieces, replacing `mem0.add()` / `mem0.search()`:

1. **`resolve_entity`** - find or create an `Entity`; near-duplicate names merge through the vector index.
2. **`retrieve_facts`** - embed the message, find the closest entities (plus the user), and traverse one hop to collect their **active** facts.
3. **`save_memories`** - one structured LLM call looks at the turn *and* the facts already known, and returns new triples plus the ids of facts that are now outdated.
4. **Storage** - new triples become `REL` edges; outdated ones are closed (`active=false`, `valid_to`), never deleted.

In [ ]:
import uuid
from pydantic import BaseModel, Field

MERGE_THRESHOLD = 0.92   # Neo4j cosine score is (1 + cos) / 2, so 0.92 ~ cosine 0.84. Raise it to merge less.


class Triple(BaseModel):
    subject: str = Field(description="Entity the fact is about. Use the speaker's exact name for first-person facts.")
    subject_type: str = Field(description="e.g. Person, Team, Project, Task, Preference, Tool")
    predicate: str = Field(description="UPPER_SNAKE_CASE relation, e.g. MEMBER_OF, WORKS_ON, PREFERS, WAITING_ON")
    object: str = Field(description="Entity or value the subject is related to")
    object_type: str
    fact: str = Field(description="The fact as one short sentence")


class MemoryUpdate(BaseModel):
    new_facts: list[Triple] = Field(description="Durable facts worth remembering from this turn; empty if none")
    outdated_fact_ids: list[str] = Field(description="ids of KNOWN facts that this turn contradicts or replaces; empty if none")


extractor = llm.with_structured_output(MemoryUpdate)

EXTRACT_PROMPT = (
    "You maintain a long-term memory graph for a ticket assistant. From the latest turn, extract DURABLE facts about the "
    "user as (subject, predicate, object) triples: who they are, their team, preferences, tasks assigned to them, people "
    "they depend on or work with, decisions. Rules:\n"
    "- Use the speaker's exact name for first-person facts; reuse the exact entity names from the known facts when they match.\n"
    "- Do NOT store ticket ids or live ticket status (they live in the ticket graph and change constantly).\n"
    "- Do NOT repeat a fact that is already known.\n"
    "- If a known fact is now false or replaced (e.g. the user changed teams), put its id in outdated_fact_ids and add the new fact.\n"
    "- Small talk and questions produce no facts."
)


def _key(name: str) -> str:
    return " ".join(name.lower().split())


def resolve_entity(name: str, etype: str) -> str:
    """Return the id of the Entity for `name`, creating it or merging it into a near-duplicate."""
    eid = f"{USER_ID}:{_key(name)}"
    if cypher("MATCH (e:Entity {id: $id}) RETURN e.id AS id", id=eid):
        return eid
    vec = embeddings.embed_query(name)
    near = cypher(
        """
        CALL db.index.vector.queryNodes('entity_embedding', 5, $vec) YIELD node, score
        WHERE node.user_id = $u AND score >= $thr
        RETURN node.id AS id ORDER BY score DESC LIMIT 1
        """,
        vec=vec, u=USER_ID, thr=MERGE_THRESHOLD,
    )
    if near:
        return near[0]["id"]
    cypher(
        """
        CREATE (e:Entity {id: $id, user_id: $u, name: $name, type: $type, created_at: datetime()})
        WITH e CALL db.create.setNodeVectorProperty(e, 'embedding', $vec)
        """,
        id=eid, u=USER_ID, name=name, type=etype, vec=vec,
    )
    return eid


def retrieve_facts(query: str, k: int = 3, limit: int = 20) -> list[dict]:
    """Facts (active edges) around the user and the entities closest to the query."""
    ids = {f"{USER_ID}:{_key(USER_NAME)}"}
    near = cypher(
        """
        CALL db.index.vector.queryNodes('entity_embedding', 10, $vec) YIELD node, score
        WHERE node.user_id = $u
        RETURN node.id AS id ORDER BY score DESC LIMIT $k
        """,
        vec=embeddings.embed_query(query), u=USER_ID, k=k,
    )
    ids |= {r["id"] for r in near}
    return cypher(
        """
        MATCH (e:Entity)-[r:REL {active: true}]-(:Entity)
        WHERE e.id IN $ids
        RETURN DISTINCT r.id AS id, r.fact AS fact, r.valid_from AS since
        ORDER BY since DESC LIMIT $limit
        """,
        ids=list(ids), limit=limit,
    )


def retrieve_memories(query: str) -> str:
    try:
        facts = retrieve_facts(query)
    except Exception as e:
        print(f"Error retrieving memories: {e}")
        return "(none)"
    print(f"   [retrieve] {len(facts)} facts")
    return "\n".join(f"- {f['fact']}" for f in facts) or "(none yet)"


def store_fact(t: Triple, episode_id: str):
    s = resolve_entity(t.subject, t.subject_type)
    o = resolve_entity(t.object, t.object_type)
    cypher(
        """
        MATCH (s:Entity {id: $s}), (o:Entity {id: $o}), (ep:Episode {id: $ep})
        MERGE (s)-[r:REL {predicate: $pred, active: true}]->(o)
          ON CREATE SET r.id = $rid, r.fact = $fact, r.valid_from = datetime(), r.source = $ep
          ON MATCH SET r.last_seen = datetime()
        MERGE (ep)-[:MENTIONS]->(s)
        MERGE (ep)-[:MENTIONS]->(o)
        """,
        s=s, o=o, ep=episode_id, pred=t.predicate.strip().upper().replace(" ", "_"), rid=uuid.uuid4().hex, fact=t.fact,
    )


def save_memories(user_msg: str, tool_calls: list[str], reply: str):
    assistant_text = reply if not tool_calls else f"(tools used: {'; '.join(tool_calls)})\n{reply}"
    try:
        known = retrieve_facts(user_msg)
        known_text = "\n".join(f"[{f['id']}] {f['fact']}" for f in known) or "(none)"
        update = extractor.invoke([
            SystemMessage(content=EXTRACT_PROMPT),
            HumanMessage(content=f"Speaker: {USER_NAME}\n\nKnown facts:\n{known_text}\n\nUser said: {user_msg}\nAssistant replied: {assistant_text}"),
        ])
        episode_id = uuid.uuid4().hex
        cypher(
            "CREATE (:Episode {id: $id, user_id: $u, text: $text, created_at: datetime()})",
            id=episode_id, u=USER_ID, text=f"User: {user_msg}\nAssistant: {assistant_text}",
        )
        known_ids = {f["id"] for f in known}
        for fid in update.outdated_fact_ids:
            if fid in known_ids:   # only close facts we actually showed the model
                cypher("MATCH ()-[r:REL {id: $id}]->() SET r.active = false, r.valid_to = datetime()", id=fid)
        for t in update.new_facts:
            store_fact(t, episode_id)
        print(f"   [memory] +{len(update.new_facts)} facts, {len(update.outdated_fact_ids)} closed")
    except Exception as e:
        print(f"Error saving memories: {e}")


def show_memories(include_history: bool = False):
    rows = cypher(
        """
        MATCH (s:Entity {user_id: $u})-[r:REL]->(o:Entity)
        WHERE $hist OR r.active
        RETURN s.name AS s, r.predicate AS p, o.name AS o, r.active AS active
        ORDER BY r.valid_from
        """,
        u=USER_ID, hist=include_history,
    )
    for r in rows:
        print(f"- {r['s']} -[{r['p']}]-> {r['o']}" + ("" if r["active"] else "   (outdated)"))
    if not rows:
        print("(no memories)")

## 5. Build the agent and one chat turn

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=(
        "You are a concise ticket assistant. The user tells you tasks assigned to them and you track them in a Neo4j graph "
        "database using your tools: create tickets for new tasks, update status as they report progress, use block_ticket "
        "when something is waiting on a person, add notes, and answer questions about their tickets. The graph tickets are "
        "the source of truth for ticket state - always use tools to change or read them; never guess ids or status. "
        "Memory context (about the user and past conversations) is background only - use it to personalise answers, but "
        "trust the live ticket list over memory if they disagree. Keep replies short."
    ),
)

short_term: list = []
SHORT_TERM_TURNS = 2


def chat_turn(user_msg: str) -> str:
    memories = retrieve_memories(user_msg)
    context = (
        f"## What we know about the user (memory graph)\n{memories}\n\n"
        f"## Live tickets (from Neo4j - source of truth)\n{tickets_snapshot()}"
    )
    messages = [SystemMessage(content=context)] + short_term + [HumanMessage(content=user_msg)]
    result = agent.invoke({"messages": messages})
    reply = result["messages"][-1].content
    tool_calls = show_tool_calls(result["messages"][len(messages):])

    short_term.extend([HumanMessage(content=user_msg), AIMessage(content=reply)])
    del short_term[:-2 * SHORT_TERM_TURNS]

    save_memories(user_msg, tool_calls, reply)
    return reply

## 6. Chat loop

Use the same script as the other notebooks so they can be compared side by side:

1. *"Hi, I'm Shiva, I'm on the payments team. I prefer short answers."*
2. *"I've been assigned: fix the checkout timeout bug (high priority) and write the refund API docs."*
3. *"Started on the checkout bug."*
4. *"Docs are blocked, waiting for the API spec from Ravi."*  -> `BLOCKED_BY` edge in the ticket graph **and** a `WAITING_ON` fact in the memory graph
5. *"What's on my plate?"*
6. *"Actually I moved to the platform team."*  -> the old `payments` fact is closed (`active=false`), a new one is added

Commands: `/tickets` (ticket graph), `/memories` (active facts in the memory graph), `/history` (including outdated facts), `quit`.

In [ ]:
print("Ticket agent ready. Commands: /tickets  /memories  /history  quit\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "bye"):
        print("Agent: Bye! Your tickets and memories are saved.")
        break
    if user_input == "/tickets":
        show_tickets()
        continue
    if user_input == "/memories":
        show_memories()
        continue
    if user_input == "/history":
        show_memories(include_history=True)
        continue
    print(f"You: {user_input}")
    print(f"Agent: {chat_turn(user_input)}\n")

## 7. Inspect what was stored

Both graphs live in Neo4j, so relationship questions are single queries.

In [ ]:
print("=== Ticket graph: tickets ===")
show_tickets()

print("\n=== Ticket graph: who is blocking my work? ===")
for r in cypher(
    "MATCH (u:User {id: $u})-[:ASSIGNED]->(t:Ticket)-[:BLOCKED_BY]->(p:Person) "
    "RETURN p.name AS person, collect('#' + toString(t.id) + ' ' + t.title) AS tickets",
    u=USER_ID,
):
    print(f"- {r['person']}: {r['tickets']}")

print("\n=== Memory graph: facts (with history) ===")
show_memories(include_history=True)

print("\n=== Memory graph: everything connected to the user within 2 hops ===")
for r in cypher(
    "MATCH (me:Entity {id: $id})-[:REL*1..2]-(x:Entity) WHERE x <> me RETURN DISTINCT x.name AS name, x.type AS type",
    id=f"{USER_ID}:{_key(USER_NAME)}",
):
    print(f"- {r['name']} ({r['type']})")

print("\n=== Memory graph: raw episodes ===")
for r in cypher("MATCH (e:Episode {user_id: $u}) RETURN e.text AS text ORDER BY e.created_at", u=USER_ID):
    print(r["text"].replace("\n", " | ")[:160])

### See the graph

In the Neo4j Browser (http://localhost:7475) run:

```cypher
MATCH (n) WHERE NOT n:Counter RETURN n
```

Memory only: `MATCH (n) WHERE n:Entity OR n:Episode RETURN n`. Ticket graph only: `MATCH (n) WHERE n:User OR n:Ticket OR n:Event OR n:Person RETURN n`.

## 8. Reset (optional)

In [ ]:
RESET = False   # set to True to wipe both graphs (tickets and memory)

if RESET:
    cypher("MATCH (n) DETACH DELETE n")
    short_term.clear()
    print("Reset done")

## Takeaways

- Neo4j now does the job of Postgres **and** Mem0: one database, two graphs - exact ticket state and long-term memory.
- What you hand-build (and Mem0 hid): triple extraction, reconciling against known facts, entity resolution, and retrieval. Each is a place where quality can drift, so debug by asking: was it extracted? merged into the wrong entity? closed as outdated?
- Vector search finds *which* entities matter; the graph then supplies their connected facts. That hybrid is the advantage over a flat list of memories - and `/history` shows how the user's situation changed because outdated facts are closed, not deleted.
- Cost: every turn adds an extraction LLM call plus embeddings, and `MERGE_THRESHOLD` needs tuning for your data.
- Close the connection when done: `driver.close()`.